# golars from Go, and back

Go cells can use the [golars](https://github.com/Gaurav-Gosain/golars) DataFrame library directly: a trailing DataFrame (or `nb.Display(df)`) is drawn as a table, and saved as HTML for Jupyter. glr cells (switch a cell with `l`, or start it with `%%glr`) run golars scripts, and frames move between the two through Arrow IPC files: `%export NAME` in glr and `golars.ReadIPC(nb.BridgePath(NAME))` in Go, `golars.WriteIPC(df, nb.BridgePath(NAME))` in Go and `%import NAME` in glr.

Tables need a golars with `DataFrame.MimeBundle`. To build against a local checkout without the network, start gopyter with `GOLARS_DIR=/path/to/golars`.

In [ ]:
import "github.com/Gaurav-Gosain/golars"

// must stops the cell on an error.
func must[T any](v T, err error) T {
	if err != nil {
		panic(err)
	}
	return v
}

var sales = must(golars.FromMap(map[string]any{
	"region": []string{"eu", "us", "apac", "us", "eu", "apac"},
	"units":  []int64{3, 5, 2, 7, 1, 4},
	"price":  []float64{4.5, 9.99, 120, 4.5, 255, 11.5},
}, []string{"region", "units", "price"}))

sales

Top-level `var`s like `sales` are recomputed in every later cell, since each cell is a new program. A DataFrame declared with `:=` isn't kept between cells.

In [ ]:
import "context"

var units = must(golars.Lazy(sales).
	GroupBy("region").
	Agg(golars.Sum("units").Alias("units")).
	Sort("units", true).
	Collect(context.Background()))

nb.Display(must(sales.Column("price")))
units

## To glr

Publish `units` for glr cells.

In [ ]:
if err := golars.WriteIPC(units, nb.BridgePath("units")); err != nil {
	panic(err)
}

In [ ]:
# A glr cell in a Go notebook: its language is in the cell's metadata.
load orders.csv
with revenue = unit_price * qty
groupby region revenue:sum:revenue
%import units
join units on region
sort revenue desc
%export summary

## Back to Go

In [ ]:
var summary = must(golars.ReadIPC(nb.BridgePath("summary")))

summary